Bronze Layer: Register raw CSV files from S3 in Databricks

In [0]:
-- Namespace Setup
CREATE CATALOG IF NOT EXISTS olist_maas_pipeline;
CREATE SCHEMA IF NOT EXISTS olist_maas_pipeline.bronze;

In [0]:
-- Set session context
USE CATALOG olist_maas_pipeline;
USE SCHEMA bronze;

In [0]:
-- SALES DOMAIN

-- Customers
CREATE OR REPLACE TABLE olist_maas_pipeline.bronze.sales_customers
USING DELTA
AS
SELECT
  CAST(customer_id AS STRING) AS customer_id,
  CAST(customer_unique_id AS STRING) AS customer_unique_id,
  CAST(customer_zip_code_prefix AS STRING) AS customer_zip_code_prefix,
  CAST(customer_city AS STRING) AS customer_city,
  CAST(customer_state AS STRING) AS customer_state,
  current_timestamp() AS ingestion_timestamp
FROM read_files(
  's3://olist-maas-landing-442186c8/raw/sales/olist_customers_dataset.csv',
  format => 'csv',
  header => true,
  inferSchema => false
);

-- Geolocation
CREATE OR REPLACE TABLE olist_maas_pipeline.bronze.sales_geolocation
USING DELTA
AS
SELECT
  CAST(geolocation_zip_code_prefix AS STRING) AS geolocation_zip_code_prefix,
  CAST(geolocation_lat AS DOUBLE) AS geolocation_lat,
  CAST(geolocation_lng AS DOUBLE) AS geolocation_lng,
  CAST(geolocation_city AS STRING) AS geolocation_city,
  CAST(geolocation_state AS STRING) AS geolocation_state,
  current_timestamp() AS ingestion_timestamp
FROM read_files(
  's3://olist-maas-landing-442186c8/raw/sales/olist_geolocation_dataset.csv',
  format => 'csv',
  header => true,
  inferSchema => false
);

-- order items
CREATE OR REPLACE TABLE olist_maas_pipeline.bronze.sales_order_items
USING DELTA
AS
SELECT
  CAST(order_id AS STRING) AS order_id,
  CAST(order_item_id AS STRING) AS order_item_id,
  CAST(product_id AS STRING) AS product_id,
  CAST(seller_id AS STRING) AS seller_id,
  try_cast(shipping_limit_date AS TIMESTAMP) AS shipping_limit_date,
  try_cast(price AS DECIMAL(18, 2)) AS price,
  try_cast(freight_value AS DECIMAL(18, 2)) AS freight_value,
  current_timestamp() AS ingestion_timestamp
FROM read_files(
  's3://olist-maas-landing-442186c8/raw/sales/olist_order_items_dataset.csv',
  format => 'csv',
  header => true,
  inferSchema => false
);

-- Order payments
CREATE OR REPLACE TABLE olist_maas_pipeline.bronze.sales_order_payments
USING DELTA
AS
SELECT
  CAST(order_id AS STRING) AS order_id,
  try_cast(payment_sequential AS INT) AS payment_sequential,
  CAST(payment_type AS STRING) AS payment_type,
  try_cast(payment_installments AS INT) AS payment_installments,
  try_cast(payment_value AS DECIMAL(18, 2)) AS payment_value,
  current_timestamp() AS ingestion_timestamp
FROM read_files(
  's3://olist-maas-landing-442186c8/raw/sales/olist_order_payments_dataset.csv',
  format => 'csv',
  header => true,
  inferSchema => false
);

-- Order reviews
CREATE OR REPLACE TABLE olist_maas_pipeline.bronze.sales_order_reviews
USING DELTA
AS
SELECT
  CAST(review_id AS STRING) AS review_id,
  CAST(order_id AS STRING) AS order_id,
  try_cast(review_score AS INT) AS review_score,
  CAST(review_comment_title AS STRING) AS review_comment_title,
  CAST(review_comment_message AS STRING) AS review_comment_message,
  try_cast(review_creation_date AS TIMESTAMP) AS review_creation_date,
  try_cast(review_answer_timestamp AS TIMESTAMP) AS review_answer_timestamp,
  current_timestamp() AS ingestion_timestamp
FROM read_files(
  's3://olist-maas-landing-442186c8/raw/sales/olist_order_reviews_dataset.csv',
  format => 'csv',
  header => true,
  inferSchema => false
);

-- Orders
CREATE OR REPLACE TABLE olist_maas_pipeline.bronze.sales_orders
USING DELTA
AS
SELECT
  CAST(order_id AS STRING) AS order_id,
  CAST(customer_id AS STRING) AS customer_id,
  CAST(order_status AS STRING) AS order_status,
  try_cast(order_purchase_timestamp AS TIMESTAMP) AS order_purchase_timestamp,
  try_cast(order_approved_at AS TIMESTAMP) AS order_approved_at,
  try_cast(order_delivered_carrier_date AS TIMESTAMP) AS order_delivered_carrier_date,
  try_cast(order_delivered_customer_date AS TIMESTAMP) AS order_delivered_customer_date,
  try_cast(order_estimated_delivery_date AS TIMESTAMP) AS order_estimated_delivery_date,
  current_timestamp() AS ingestion_timestamp
FROM read_files(
  's3://olist-maas-landing-442186c8/raw/sales/olist_orders_dataset.csv',
  format => 'csv',
  header => true,
  inferSchema => false
);

-- Products
CREATE OR REPLACE TABLE olist_maas_pipeline.bronze.sales_products
USING DELTA
AS
SELECT
  CAST(product_id AS STRING) AS product_id,
  CAST(product_category_name AS STRING) AS product_category_name,
  try_cast(product_name_lenght AS INT) AS product_name_length,
  try_cast(product_description_lenght AS INT) AS product_description_length,
  try_cast(product_photos_qty AS INT) AS product_photos_qty,
  CAST(product_weight_g AS DOUBLE) AS product_weight_g,
  CAST(product_length_cm AS DOUBLE) AS product_length_cm,
  CAST(product_height_cm AS DOUBLE) AS product_height_cm,
  CAST(product_width_cm AS DOUBLE) AS product_width_cm,
  current_timestamp() AS ingestion_timestamp
FROM read_files(
  's3://olist-maas-landing-442186c8/raw/sales/olist_products_dataset.csv',
  format => 'csv',
  header => true,
  inferSchema => false
);

-- Sellers
CREATE OR REPLACE TABLE olist_maas_pipeline.bronze.sales_sellers
USING DELTA
AS
SELECT
  CAST(seller_id AS STRING) AS seller_id,
  CAST(seller_zip_code_prefix AS STRING) AS seller_zip_code_prefix,
  CAST(seller_city AS STRING) AS seller_city,
  CAST(seller_state AS STRING) AS seller_state,
  current_timestamp() AS ingestion_timestamp
FROM read_files(
  's3://olist-maas-landing-442186c8/raw/sales/olist_sellers_dataset.csv',
  format => 'csv',
  header => true,
  inferSchema => false
);

-- Category translation
CREATE OR REPLACE TABLE olist_maas_pipeline.bronze.sales_product_category_translation
USING DELTA
AS
SELECT
  CAST(product_category_name AS STRING) AS product_category_name,
  CAST(product_category_name_english AS STRING) AS product_category_name_english,
  current_timestamp() AS ingestion_timestamp
FROM read_files(
  's3://olist-maas-landing-442186c8/raw/sales/product_category_name_translation.csv',
  format => 'csv',
  header => true,
  inferSchema => false
);

In [0]:
-- MARKETING DOMAIN

-- Closed deals
CREATE OR REPLACE TABLE olist_maas_pipeline.bronze.marketing_closed_deals
USING DELTA
AS
SELECT
  CAST(mql_id AS STRING) AS mql_id,
  CAST(seller_id AS STRING) AS seller_id,
  CAST(sdr_id AS STRING) AS sdr_id,
  CAST(sr_id AS STRING) AS sr_id,
  try_cast(won_date AS TIMESTAMP) AS won_date,
  CAST(business_segment AS STRING) AS business_segment,
  CAST(lead_type AS STRING) AS lead_type,
  CAST(lead_behaviour_profile AS STRING) AS lead_behaviour_profile,
  CAST(has_company AS STRING) AS has_company,
  CAST(has_gtin AS STRING) AS has_gtin,
  CAST(average_stock AS STRING) AS average_stock,
  CAST(business_type AS STRING) AS business_type,
  try_cast(declared_product_catalog_size AS INT) AS declared_product_catalog_size,
  try_cast(declared_monthly_revenue AS DECIMAL(18,2)) AS declared_monthly_revenue,
  current_timestamp() AS ingestion_timestamp
FROM read_files(
  's3://olist-maas-landing-442186c8/raw/marketing/olist_closed_deals_dataset.csv',
  format => 'csv',
  header => true,
  inferSchema => false
);

--- Marketing qualified lead
CREATE OR REPLACE TABLE olist_maas_pipeline.bronze.marketing_mql
USING DELTA
AS
SELECT
  CAST(mql_id AS STRING) AS mql_id,
  try_cast(first_contact_date AS TIMESTAMP) AS first_contact_date,
  CAST(landing_page_id AS STRING) AS landing_page_id,
  CAST(origin AS STRING) AS origin,
  current_timestamp() AS ingestion_timestamp
FROM read_files(
  's3://olist-maas-landing-442186c8/raw/marketing/olist_marketing_qualified_leads_dataset.csv',
  format => 'csv',
  header => true,
  inferSchema => false
);

In [0]:
-- TESTING DOMAIN

--  A/B testing
CREATE OR REPLACE TABLE olist_maas_pipeline.bronze.testing_marketing_ab
USING DELTA
AS
SELECT
  CAST(`user id` AS STRING) AS user_id,
  CAST(`test group` AS STRING) AS test_group,
  CAST(converted AS STRING) AS converted,
  try_cast(`total ads` AS INT) AS total_ads,
  CAST(`most ads day` AS STRING) AS most_ads_day,
  try_cast(`most ads hour` AS INT) AS most_ads_hour,
  current_timestamp() AS ingestion_timestamp
FROM read_files(
  's3://olist-maas-landing-442186c8/raw/testing/marketing_AB.csv',
  format => 'csv',
  header => true,
  inferSchema => false
);

In [0]:
-- Confirm all tables exist
SHOW TABLES IN olist_maas_pipeline.bronze;